# 03. A/B Testing & Statistical Hypothesis Testing
This notebook evaluates the statistical significance of promotions on Customer Conversion (Repeat Purchase Rate), AOV, and Revenue per Customer.

## Framework
- **Significance level (\(\alpha\))**: 0.05
- **Assumptions**: Independent observations; large sample sizes for Z-test; normality/equal variances (or robustness to deviations due to large N) for t-test.
- **Hypotheses for Conversion Rate**:
  - \(H_0\): \(P_{promo} = P_{control}\) (Promotions do not increase repeat purchase conversion)
  - \(H_1\): \(P_{promo} > P_{control}\) (Promotions increase repeat purchase conversion)

In [ ]:
import pathlib
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

base_dir = pathlib.Path().resolve().parent
processed_dir = base_dir / 'data' / 'processed'
df_orders = pd.read_csv(processed_dir / 'processed_orders.csv')

# Split into Treatment (any promotion P1, P2, P3) and Control (CONTROL)
treatment = df_orders[df_orders['promotion_flag'] == 1]
control = df_orders[df_orders['promotion_flag'] == 0]

## 1. Repeat Purchase Conversion Rate Z-Test

In [ ]:
# Conversion is defined as the customer becoming a repeat customer
repeat_treatment = treatment.groupby('customer_unique_id')['repeat_customer_flag'].max()
repeat_control = control.groupby('customer_unique_id')['repeat_customer_flag'].max()

successes = np.array([repeat_treatment.sum(), repeat_control.sum()])
nobs = np.array([len(repeat_treatment), len(repeat_control)])

# Run 1-tailed proportions z-test (alternative='larger' for treatment > control)
z_stat, p_val = proportions_ztest(successes, nobs, alternative='larger')
effect_size = (repeat_treatment.mean() - repeat_control.mean())

print(f'Z-Statistic: {z_stat:.4f}')
print(f'P-Value: {p_val:.4f}')
print(f'Absolute conversion difference: {effect_size:.4f}')

## 2. Average Order Value (AOV) Independent t-test

In [ ]:
aov_treatment = treatment['order_value']
aov_control = control['order_value']

t_stat, t_pval = stats.ttest_ind(aov_treatment, aov_control, equal_var=False)

# Calculate Cohen's d for effect size
def cohen_d(x, y):
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / dof)
    return (np.mean(x) - np.mean(y)) / pooled_std

d_val = cohen_d(aov_treatment, aov_control)

print(f'AOV t-statistic: {t_stat:.4f}')
print(f'AOV p-value: {t_pval:.4f}')
print(f'AOV Cohen\'s d: {d_val:.4f}')

## 3. Revenue per Customer Independent t-test

In [ ]:
rev_cust_treatment = treatment.groupby('customer_unique_id')['order_value'].sum()
rev_cust_control = control.groupby('customer_unique_id')['order_value'].sum()

t_stat_rev, t_pval_rev = stats.ttest_ind(rev_cust_treatment, rev_cust_control, equal_var=False)
d_val_rev = cohen_d(rev_cust_treatment, rev_cust_control)

print(f'Revenue per Customer t-statistic: {t_stat_rev:.4f}')
print(f'Revenue per Customer p-value: {t_pval_rev:.4f}')
print(f'Revenue per Customer Cohen\'s d: {d_val_rev:.4f}')

## Business Implications & Guidelines
- **If p-value < 0.05**: We reject the null hypothesis and conclude that promotions have a statistically significant positive effect on the metric.
- **If p-value >= 0.05**: We fail to reject the null hypothesis, suggesting that promotions do not show a statistically significant difference over the control cohort.